# Audio Transcription Using Full and Quantized Open-Source Models

### Import Required Libraries

In [1]:
import librosa
import os
import torch
import time
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

### Load Audio

In [2]:
audio, sr = librosa.load("Documents/AIML-Assessment/Module-4/speech.wav", sr=16000)
print(audio.shape)

(293700,)


### Normalize

In [3]:
audio = audio / abs(audio).max()

### Loading Full Model

In [4]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

### Create Quantized Model

In [5]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

C:\Users\acer\AppData\Local\Temp\ipykernel_25572\2285888857.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(
C:\Users\acer\anaconda3\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that pro

### Transcription

In [6]:
def transcribe(audio, model, processor):
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    start = time.time()
    predicted_ids = model.generate(inputs.input_features)
    inference_time = time.time() - start
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription, inference_time

### Full Model Transcription Results

In [7]:
full_text, full_time = transcribe(audio, model, processor)
print(full_text)
print("Inference Time:", round(full_time, 3), "seconds")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

 The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health in zest. A salt pickle tastes fine with ham. Tacos all pastora are my favorite. A zestful food is the hot cross bun.
Inference Time: 3.745 seconds


### Quantized Model Transcription Results

In [8]:
quant_text, quant_time = transcribe(audio, quantized_model, processor)
print(quant_text)
print("Inference Time:", round(quant_time, 3), "seconds")

 The snail smell of old beer lingers. It takes heat to bring out the odor. I called dip restores health and zest. A salt pickle tastes fine with him. Talk to those all pastora, are my favorite. A zestful food is the hot cross-bun.
Inference Time: 2.42 seconds


### Original Text

In [9]:
original_text = """
The stale smell of old beer lingers.
It takes heat to bring out the odor.
A cold dip restores health in zest.
A salt pickle tastes fine with ham.
Tacos al pastor are my favorite.
A zestful food is the hot cross bun.
""".strip()

### Calculate WER

In [10]:
full_wer = wer(original_text, full_text)
quant_wer = wer(original_text, quant_text)
print("Full Model WER:", round(full_wer, 4))
print("Quantized Model WER:", round(quant_wer, 4))

Full Model WER: 0.3158
Quantized Model WER: 0.5


###  Calculate CER

In [11]:
full_cer = cer(original_text,full_text)
quant_cer = cer(original_text,quant_text)
print("Full Model CER:", round(full_cer, 4))
print("Quantized Model CER:", round(quant_cer, 4))

Full Model CER: 0.0326
Quantized Model CER: 0.1302


### Model Size

In [12]:
def model_size_mb(model):
    total_bytes = 0
    for param in model.parameters():
        total_bytes += param.nelement() * param.element_size()
    return total_bytes / (1024 ** 2)

In [13]:
print("Full Model Size:", round(model_size_mb(model), 2), "MB")
print("Full Model Size:", round(model_size_mb(quantized_model), 2), "MB")

Full Model Size: 144.05 MB
Full Model Size: 80.93 MB
